# NB_patient_360

Builds `adb_caresync_project.gold.patient_360` (one row per patient) from silver patients, hospitals, insurance_providers and aggregated lab_results, per STTM Silver to Gold v1.0. Load type: full rebuild (CREATE OR REPLACE TABLE AS SELECT).

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS adb_caresync_project.gold;

CREATE OR REPLACE TABLE adb_caresync_project.gold.patient_360 AS
WITH lab_agg AS (
  -- Aggregate lab results to patient grain
  SELECT
    patient_id,
    COUNT(*)                              AS total_tests,
    COUNT_IF(result_status = 'Abnormal')  AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')  AS critical_test_count,
    MAX(test_date)                        AS last_test_date
  FROM adb_caresync_project.silver.lab_results
  GROUP BY patient_id
)
SELECT
  CAST(p.patient_id AS INT)                                                     AS patient_id,
  CAST(p.first_name AS STRING)                                                  AS first_name,
  CAST(p.last_name AS STRING)                                                   AS last_name,
  CAST(p.date_of_birth AS DATE)                                                 AS date_of_birth,
  CAST(FLOOR(datediff(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  CAST(p.gender AS STRING)                                                      AS gender,
  CAST(p.city AS STRING)                                                        AS city,
  CAST(p.state AS STRING)                                                       AS state,
  CAST(p.registered_hospital_id AS INT)                                         AS hospital_id,
  CAST(h.hospital_name AS STRING)                                               AS hospital_name,
  CAST(p.insurance_provider_id AS INT)                                          AS insurance_provider_id,
  CAST(ip.provider_name AS STRING)                                              AS insurance_provider_name,
  CAST(COALESCE(la.total_tests, 0) AS INT)                                      AS total_tests,
  CAST(COALESCE(la.abnormal_test_count, 0) AS INT)                              AS abnormal_test_count,
  CAST(COALESCE(la.critical_test_count, 0) AS INT)                              AS critical_test_count,
  CAST(la.last_test_date AS DATE)                                               AS last_test_date
FROM adb_caresync_project.silver.patients p
LEFT JOIN adb_caresync_project.silver.hospitals h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN adb_caresync_project.silver.insurance_providers ip
  ON p.insurance_provider_id = ip.insurance_provider_id
LEFT JOIN lab_agg la
  ON p.patient_id = la.patient_id;

In [0]:
updated_df = spark.table("adb_caresync_project.gold.patient_360")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))